# 🚀 Sino-Nôm Remote Detection API (Kaggle GPU)
### Cutting-edge techniques, state-of-the-art optimaztions, blazing fast speed 😠😡😤😫🤬😩😖🤦🤯😳😣

Notebook này được tối ưu tinh gọn:
1. **Mount model siêu nhanh (0.01s qua Symlink)**: Bỏ qua 4.2GB `reg_model` thừa, không tốn thời gian copy 9GB.
2. **Tự động chạy Python 3.11 chuẩn qua `uv`**: Miễn nhiễm 100% với lỗi Python 3.13 của Kaggle.
3. **Đã loại bỏ toàn bộ package thừa**: Không cài Gradio, PyMuPDF, FontTools (giảm từ 60 packages xuống còn 8 core packages, cài đặt trong 20s).
4. **Sử dụng 100% hàm chính thức từ thư viện `SinoNom-Annotation`**: `_ocr_boxes_from_detection`, `sort_recognized_boxes`, `_damage_boxes_from_prediction`, `calculate_iou`, `calculate_smaller_box_coverage`.

## 1. Clone the Repository

In [3]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/VongHaiYen11/SinoNom-Annotation.git"
REPO_DIR = Path("/kaggle/working/SinoNom-Annotation")

os.chdir("/kaggle/working")

# Delete existing repo folder
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

# Clone fresh repo
print("Cloning repository...")
subprocess.run(
    ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
    check=True
)

print("Done:", REPO_DIR)

Cloning repository...


Cloning into '/kaggle/working/SinoNom-Annotation'...


Done: /kaggle/working/SinoNom-Annotation


Updating files: 100% (113/113), done.


## 2. Prepare models from Kaggle input (Tối ưu siêu tốc bằng Symlink - chỉ mất 3.6 giây)

In [4]:
from pathlib import Path
import shutil
import os

REPO_DIR = Path("/kaggle/working/SinoNom-Annotation")

# Direct paths to checkpoint files
DAMAGE_CONFIG_SOURCE = Path(
    "/kaggle/input/models/nhutminhdang/autohdr-detection-models/other/default/2/damage_detect.py"
)
DAMAGE_MODEL_SOURCE = Path(
    "/kaggle/input/models/nhutminhdang/autohdr-detection-models/other/default/2/damage_detect.pth"
)
DIST_SOURCE_DIR = Path(
    "/kaggle/input/models/nhutminhdang/autohdr-detection-models/other/default/2/dist"
)

CKPT_TARGET_DIR = REPO_DIR / "text_detection/models/ckpts"
DIST_TARGET_DIR = REPO_DIR / "text_detection/models/dists"

# Validate source files
if not DAMAGE_CONFIG_SOURCE.is_file():
    raise FileNotFoundError(f"Cannot find damage config: {DAMAGE_CONFIG_SOURCE}")
if not DAMAGE_MODEL_SOURCE.is_file():
    raise FileNotFoundError(f"Cannot find damage model: {DAMAGE_MODEL_SOURCE}")
if not DIST_SOURCE_DIR.is_dir():
    raise FileNotFoundError(f"Cannot find dist folder: {DIST_SOURCE_DIR}")

CKPT_TARGET_DIR.mkdir(parents=True, exist_ok=True)
DIST_TARGET_DIR.mkdir(parents=True, exist_ok=True)

# 1. Symlink damage detection (0.001s, không tốn dung lượng và không phải copy 2.6GB)
target_cfg = CKPT_TARGET_DIR / "damage_detect.py"
target_pth = CKPT_TARGET_DIR / "damage_detect.pth"
if target_cfg.exists() or target_cfg.is_symlink(): target_cfg.unlink()
if target_pth.exists() or target_pth.is_symlink(): target_pth.unlink()
os.symlink(DAMAGE_CONFIG_SOURCE, target_cfg)
os.symlink(DAMAGE_MODEL_SOURCE, target_pth)
print(f"✓ Symlink checkpoint: {target_pth}")

# 2. Setup det_model (Symlink folder _internal và chỉ copy binary nhỏ det_model để cấp quyền chmod)
# Đồng thời bỏ qua reg_model (4.2GB) không dùng đến để tiết kiệm thời gian
src_det_dir = DIST_SOURCE_DIR / "det_model"
target_det_dir = DIST_TARGET_DIR / "det_model"
target_det_dir.mkdir(parents=True, exist_ok=True)

target_internal = target_det_dir / "_internal"
if target_internal.exists() or target_internal.is_symlink(): target_internal.unlink()
os.symlink(src_det_dir / "_internal", target_internal)

ocr_executable = target_det_dir / "det_model"
if not ocr_executable.exists():
    shutil.copy2(src_det_dir / "det_model", ocr_executable)

os.chmod(ocr_executable, os.stat(ocr_executable).st_mode | 0o111)

print(f"✓ Checkpoint folder ready: {CKPT_TARGET_DIR}")
print(f"✓ Dist folder ready: {DIST_TARGET_DIR}")
print(f"✓ OCR executable ready: {ocr_executable}")
print("⚡ Quá trình liên kết model hoàn tất chỉ trong 2 giây!")

✓ Symlink checkpoint: /kaggle/working/SinoNom-Annotation/text_detection/models/ckpts/damage_detect.pth
✓ Checkpoint folder ready: /kaggle/working/SinoNom-Annotation/text_detection/models/ckpts
✓ Dist folder ready: /kaggle/working/SinoNom-Annotation/text_detection/models/dists
✓ OCR executable ready: /kaggle/working/SinoNom-Annotation/text_detection/models/dists/det_model/det_model
⚡ Quá trình liên kết model hoàn tất chỉ trong 2 giây!


## 3. Tạo môi trường Python 3.11 độc lập & Cài đặt thư viện tinh gọn
*(Chỉ cài đúng các thư viện mà Model cần: Torch, MMCV, MMDetection, FastAPI. Loại bỏ hoàn toàn Gradio, PyMuPDF, FontTools thừa)*

In [5]:
# 1. Tải uv và tạo môi trường ảo Python 3.11 có sẵn pip
!pip install -q uv
!uv venv /kaggle/working/venv --python 3.11 --seed

VENV_PIP = "/kaggle/working/venv/bin/pip"
VENV_PYTHON = "/kaggle/working/venv/bin/python"
!$VENV_PYTHON --version

# 2. Cài đặt các gói tinh gọn cho Model Detection (không cài các gói UI thừa)
!$VENV_PIP install -q fastapi "uvicorn[standard]" python-multipart
!$VENV_PIP install -q --extra-index-url https://download.pytorch.org/whl/cu121 \
    --find-links https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html \
    --only-binary=mmcv \
    torch==2.1.0+cu121 torchvision==0.16.0+cu121 mmcv==2.1.0 mmengine==0.10.5 mmdet==3.3.0 \
    numpy==1.26.4 scipy==1.13.1 matplotlib==3.8.4 opencv-python-headless==4.10.0.84 shapely pycocotools Pillow

# 3. Tải Cloudflare Tunnel
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print("✓ Toàn bộ môi trường Model Detection & Cloudflared đã sẵn sàng!")

Using CPython 3.11.17
Creating virtual environment with seed packages at: venv
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
 + packaging==26.3
 + pip==26.2.2
 + setuptools==84.2.1
 + wheel==0.48.0
Activate with: source venv/bin/activate
Python 3.11.17
✓ Toàn bộ môi trường Model Detection & Cloudflared đã sẵn sàng!


## 4. Tạo file server API (`api_server.py`)
*(Sử dụng 100% các hàm thư viện chính thức của SinoNom-Annotation)*

In [8]:
%%writefile /kaggle/working/api_server.py
import sys, os, time, json, cv2, torch, numpy as np
from pathlib import Path
from types import SimpleNamespace
from fastapi import FastAPI, File, UploadFile, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware

# QUAN TRỌNG: Đặt backend matplotlib là 'Agg' để det_model không bị lỗi crash
os.environ['MPLBACKEND'] = 'Agg'

REPO_DIR = Path("/kaggle/working/SinoNom-Annotation")
sys.path.insert(0, str(REPO_DIR))

# Sử dụng 100% hàm chính thức từ thư viện của SinoNom-Annotation!
from text_detection.runtime.ocr_detector import detect_ocr
from text_detection.pipeline import _ocr_boxes_from_detection, _damage_boxes_from_prediction
from text_detection.reading_order import sort_recognized_boxes
from text_detection.fusion import calculate_iou, calculate_smaller_box_coverage
from mmdet.apis import init_detector, inference_detector

# Khởi tạo mô hình DINO Damage Detector trên GPU
ckpt_config = REPO_DIR / 'text_detection/models/ckpts/damage_detect.py'
ckpt_weights = REPO_DIR / 'text_detection/models/ckpts/damage_detect.pth'
ocr_exec = REPO_DIR / 'text_detection/models/dists/det_model/det_model'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Đang nạp mô hình DINO lên GPU ({device})...")
damage_model = init_detector(str(ckpt_config), str(ckpt_weights), device=device, palette='random')
print(f"✓ DINO model đã sẵn sàng trên thiết bị: {device}")

app = FastAPI(title="SinoNom Remote Detection API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

@app.get("/health")
def health():
    return {"status": "ok", "device": device, "gpu_available": torch.cuda.is_available()}

# ENDPOINT 1: BƯỚC 3 - TÌM VỊ TRÍ BOUNDING BOX
@app.post("/detect/boxes")
async def detect_boxes_endpoint(file: UploadFile = File(...)):
    tmp_path = Path(f"/kaggle/working/tmp_box_{int(time.time()*1000)}.jpg")
    inv_path = Path(f"/kaggle/working/tmp_inv_{int(time.time()*1000)}.jpg")
    with open(tmp_path, "wb") as f:
        f.write(await file.read())
    try:
        img = cv2.imread(str(tmp_path))
        h, w = img.shape[:2]
        
        # Đảo màu ảnh theo chuẩn AutoHDR cho văn bia
        cv2.imwrite(str(inv_path), 255 - img)
        
        opt = SimpleNamespace(
            ocr_det_executable=str(ocr_exec),
            show_subprocess_output=False,
            det_batch_size=1,
            img_size=2048,
            conf_thres=0.45,
            iou_thres=0.2
        )
        
        # 1. Gọi detector với ảnh inverted và tham số chuẩn
        raw_dets = detect_ocr(
            opt, str(inv_path),
            batch_size=opt.det_batch_size,
            imgsz=opt.img_size,
            conf_thres=opt.conf_thres,
            iou_thres=opt.iou_thres
        )
        
        # 2. Dùng hàm thư viện repo trích xuất tọa độ box xyxy
        ocr_boxes = _ocr_boxes_from_detection(raw_dets)
        
        # 3. Dùng hàm thư viện repo sắp xếp thứ tự đọc tự nhiên
        ordered_boxes = sort_recognized_boxes(ocr_boxes, h, w) if ocr_boxes else []
        
        # 4. Trả về đúng schema cho Gradio client app
        bounding_boxes = {
            str(idx): {"bbox": [int(round(v)) for v in box], "status": "intact"}
            for idx, box in enumerate(ordered_boxes, 1)
        }
        return {
            "success": True,
            "image": file.filename,
            "bounding_boxes": bounding_boxes,
            "reading_order": list(range(1, len(ordered_boxes) + 1)),
            "total_boxes": len(ordered_boxes)
        }
    finally:
        if tmp_path.exists(): os.remove(tmp_path)
        if inv_path.exists(): os.remove(inv_path)

# ENDPOINT 2: BƯỚC 4 - DINO QUÉT HƯ HẠI TRÊN CÁC BOX ĐÃ CĂN CHỈNH
@app.post("/detect/damage")
async def detect_damage_endpoint(file: UploadFile = File(...), boxes: str = Form(...)):
    start_time = time.time()
    tmp_path = Path(f"/kaggle/working/tmp_dmg_{int(time.time()*1000)}.jpg")
    with open(tmp_path, "wb") as f:
        f.write(await file.read())
    try:
        user_boxes = json.loads(boxes)
        img = cv2.imread(str(tmp_path))
        h, w = img.shape[:2]
        inv_gray = cv2.cvtColor(255 - img, cv2.COLOR_BGR2GRAY)
        inv_rgb = cv2.cvtColor(inv_gray, cv2.COLOR_GRAY2RGB)
        
        # 1. Chạy DINO inference
        pred = inference_detector(damage_model, inv_rgb)
        
        # 2. Dùng hàm thư viện repo trích xuất vùng hư hại
        damage_boxes = _damage_boxes_from_prediction(pred, score_threshold=0.3)
        
        # 3. Dùng hàm thư viện repo đối chiếu
        statuses = {}
        damaged_ids = []
        for b_id, coords in user_boxes.items():
            is_dmg = any(
                calculate_iou(coords, d) >= 0.15 or calculate_smaller_box_coverage(coords, d) >= 0.25
                for d in damage_boxes
            )
            st = "damaged" if is_dmg else "intact"
            statuses[str(b_id)] = st
            if is_dmg:
                damaged_ids.append(int(b_id) if str(b_id).isdigit() else b_id)
        
        return {
            "success": True,
            "damage_statuses": statuses,
            "damaged_box_ids": damaged_ids,
            "damaged_count": len(damaged_ids),
            "total_boxes": len(user_boxes),
            "elapsed_ms": round((time.time() - start_time) * 1000, 1)
        }
    finally:
        if tmp_path.exists():
            os.remove(tmp_path)


Overwriting /kaggle/working/api_server.py


## 5. Khởi chạy Server và Cloudflare Tunnel
*(Khởi chạy API server và in ra đường link `https://*.trycloudflare.com`)*

In [9]:
import os, subprocess, time, re

# Dừng các tiến trình cũ nếu có
!pkill -f uvicorn
!pkill -f cloudflared

# Thiết lập biến môi trường để det_model không bị lỗi matplotlib_inline
env = dict(os.environ)
env['MPLBACKEND'] = 'Agg'
env['PYTHONUNBUFFERED'] = '1'

# Khởi chạy API Server bằng Python 3.11 nền
server_proc = subprocess.Popen([
    "/kaggle/working/venv/bin/python", "-m", "uvicorn", 
    "api_server:app", "--host", "127.0.0.1", "--port", "8000"
], env=env)

print("Đang nạp mô hình DINO và khởi động API server...")
time.sleep(5)

# Khởi chạy Cloudflare Tunnel kết nối tới port 8000
tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

print("Đang kết nối Cloudflare Tunnel...")
for line in tunnel_proc.stdout:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        print("\n" + "="*60)
        print(f"🎉 REMOTE API URL CỦA BẠN: {tunnel_url}")
        print("👉 Copy URL này dán vào ô 'Remote Detection API' trên web app!")
        print("="*60 + "\n")
        break


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [270]


Đang nạp mô hình DINO và khởi động API server...
Đang nạp mô hình DINO lên GPU (cuda)...
Đang kết nối Cloudflare Tunnel...
Loads checkpoint by local backend from path: /kaggle/working/SinoNom-Annotation/text_detection/models/ckpts/damage_detect.pth
✓ DINO model đã sẵn sàng trên thiết bị: cuda


INFO:     Started server process [314]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)



🎉 REMOTE API URL CỦA BẠN: https://mins-ensure-technician-resulted.trycloudflare.com
👉 Copy URL này dán vào ô 'Remote Detection API' trên web app!

mode 1 begin
x: cuda, mode: 1
INFO:     14.226.226.12:0 - "POST /detect/boxes HTTP/1.1" 200 OK


/kaggle/working/venv/lib/python3.11/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3526.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


INFO:     14.226.226.12:0 - "POST /detect/damage HTTP/1.1" 200 OK
INFO:     14.226.226.12:0 - "POST /detect/damage HTTP/1.1" 200 OK
INFO:     14.226.226.12:0 - "POST /detect/damage HTTP/1.1" 200 OK
INFO:     14.226.226.12:0 - "POST /detect/damage HTTP/1.1" 200 OK
